# Dual-GPU Bayesian Elite — one backbone, all seeds

Notebook này chạy tuần tự mọi dataset/seed của một backbone. Trong từng run, GPU 0 xử lý TB và GPU 1 xử lý DB song song.

In [ ]:
BACKBONE = "REPLACE_BACKBONE"
GIT_COMMIT = "REPLACE_GIT_COMMIT"
SEEDS = [42, 44, 46, 48, 50]
RUNS = []
MC_SAMPLES = 32
GFLOWNET_ITERATIONS = 5000
NUM_EPOCHS = 100
PATIENCE = 5

In [ ]:
from pathlib import Path

if any(run['dataset_id'] == 'culture-b' for run in RUNS):
    source = Path('/kaggle/input/datasets/utkarshsaxenadn/fast-food-classification-dataset/Fast Food Classification V2')
    for destination, origin in {'train': 'Train', 'test': 'Test', 'val': 'Valid'}.items():
        path = Path('/kaggle/working') / destination
        if path.is_symlink():
            path.unlink()
        elif path.exists():
            raise FileExistsError(f'Không ghi đè path thật: {path}')
        path.symlink_to(source / origin, target_is_directory=True)
for run in RUNS:
    assert Path(run['data_dir']).exists(), run['data_dir']

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/khoaddb2207532/neuro_symbolic_mlops_l2_app.git'
PROJECT = Path('/kaggle/working/neuro_symbolic_mlops_l2_app')
if not PROJECT.exists():
    clone_url = REPO_URL
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('GITHUB_TOKEN')
        if token:
            clone_url = REPO_URL.replace('https://', f'https://{token}@')
    except Exception:
        pass
    subprocess.run(['git', 'clone', '--filter=blob:none', clone_url, str(PROJECT)], check=True)
subprocess.run(['git', 'fetch', 'origin', GIT_COMMIT, '--depth', '1'], cwd=PROJECT, check=True)
subprocess.run(['git', 'checkout', '--detach', GIT_COMMIT], cwd=PROJECT, check=True)
actual = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip()
assert actual == GIT_COMMIT, (actual, GIT_COMMIT)
print('Pinned commit:', actual)

In [ ]:
%cd /kaggle/working/neuro_symbolic_mlops_l2_app
!pip install -q torchgfn tensordict dvclive dvc openpyxl
!nvidia-smi -L
import torch
assert torch.cuda.device_count() >= 2, 'Hãy chọn Kaggle Accelerator: GPU T4 x2'
print('CUDA devices:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
import logging
import os
import warnings

warnings.filterwarnings('ignore')
logging.getLogger('dvclive').setLevel(logging.ERROR)
logging.getLogger('dvc').setLevel(logging.ERROR)
os.environ['PYTHONWARNINGS'] = 'ignore'
os.environ['DVCLIVE_LOGLEVEL'] = 'ERROR'
os.environ['DVC_NO_ANALYTICS'] = '1'
print('Đã tắt warning của Python, DVCLive và DVC.')

In [ ]:
import subprocess
import sys
from pathlib import Path

MODEL_ROOT = Path('/kaggle/working/dual_elite_model_runs') / BACKBONE
for index, run in enumerate(RUNS, start=1):
    run_root = MODEL_ROOT / run['dataset_id'] / f"seed_{run['seed']}"
    command = [
        sys.executable, '-m', 'pipelines.run_dual_gpu_elite_seed_experiment',
        '--config', 'params.yaml',
        '--seed', str(run['seed']),
        '--dataset-id', run['dataset_id'],
        '--prior-run-id', run['prior_run_id'],
        '--backbone', BACKBONE,
        '--data-dir', run['data_dir'],
        '--output-dir', str(run_root),
        '--project-dir', str(PROJECT),
        '--kaggle-input-root', '/kaggle/input',
        '--mc-samples', str(MC_SAMPLES),
        '--gflownet-iterations', str(GFLOWNET_ITERATIONS),
        '--num-epochs', str(NUM_EPOCHS),
        '--patience', str(PATIENCE),
    ]
    print(f"\n===== RUN {index}/{len(RUNS)}: {run['dataset_id']} seed={run['seed']} backbone={BACKBONE} =====", flush=True)
    subprocess.run(command, cwd=PROJECT, check=True)

summary_dir = Path('/kaggle/working/dual_elite_model_summaries') / BACKBONE
subprocess.run([
    sys.executable, '-m', 'pipelines.aggregate_dual_gpu_elite_model',
    '--input-root', str(MODEL_ROOT),
    '--output-dir', str(summary_dir),
    '--backbone', BACKBONE,
    '--expected-seeds', *map(str, SEEDS),
], cwd=PROJECT, check=True)

## Resume

Nếu Kaggle bị ngắt, Save Version và Add Input output đó vào lần chạy mới. Notebook lặp lại từ đầu danh sách nhưng từng run tự restore/skip stage hoàn tất và resume checkpoint còn dở.